In [ ]:
#Import necessary libraries
import sys
import os
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from pycgp_finalclass.Config import CGPConfig
from pycgp_finalclass.ES import ES
from pycgp_finalclass.Evaluator import EvaluatorSin, Binary_Classifier,Binary_Regressor, Regressor, MultiClassClassifier
from pycgp_finalclass.Node import Node
from pycgp_finalclass.Mutation import Golden_mutation
from pycgp_finalclass.Function import Func
from pycgp_finalclass.Function_library import *
from pycgp_finalclass.Genome import CGPGenome

In [ ]:
def build_funcLib(): #Define the function used
    return [Func(f_sum, 'sum', 2, 0), #Put in comments function that are not used
            Func(f_aminus, 'aminus', 2, 0),
            Func(f_mult, 'mult', 2, 0),
            Func(f_exp, 'exp', 2, 0),
            Func(f_abs, 'abs', 1, 0),
            Func(f_sqrt, 'sqrt', 1, 0),
            Func(f_sqrtxy, 'sqrtxy', 2, 0),
            Func(f_squared, 'squared', 1, 0),
            Func(f_pow, 'pow', 2, 0),
            Func(f_one, 'one', 0, 0),
            Func(f_zero, 'zero', 0, 0),
            Func(f_const, 'const', 0, 1),
            #Func(f_inv, 'inv', 1, 0),
            Func(f_gt, 'gt', 2, 0),
            Func(safe_div, 'safe_div', 2, 0),
            #Func(f_asin, 'asin', 1, 0),
            #Func(f_acos, 'acos', 1, 0),
            #Func(f_atan, 'atan', 1, 0),
            #Func(f_sin, 'sin', 1, 0),
            Func(f_min, 'min', 2, 0),
            Func(f_max, 'max', 2, 0)
            #Func(f_round, 'round', 1, 0),
            #Func(f_floor, 'floor', 1, 0),
            #Func(f_ceil, 'ceil', 1, 0)
            ]
functions = build_funcLib()

In [15]:
## Initialization of the CGP algorithm for regression ##

from pmlb import fetch_data
from sklearn.preprocessing import MinMaxScaler


# Selection of dataset
# We use a PMLB dataset for regression in this example
# You can change the dataset by modifying the fetch_data function parameters and putting the Dataset ID you want to use
gametes = fetch_data('1027_ESL')
# We save the dataset in a local directory
X, y = fetch_data('1027_ESL', return_X_y=True, local_cache_dir='./datasets')

# Float conversion data
X = X.astype(float)
y = y.astype(float)

# Standardize the data between -1 and 1
# This is important for the CGP algorithm to work properly
scaler = MinMaxScaler(feature_range=(-1, 1))
X = scaler.fit_transform(X)



# Here we define the evaluator for the CGP algorithm
# In this case we use a regression evaluator but you can use a multi-class classifier in the case of classification problems
evaluator = Regressor(X, y,cv=True)

#Initialise config with the same number of inputs as features in the datasets, internal nodes around 20/30, outputs(number of classes)
CGP_config = CGPConfig(num_inputs=X.shape[1], num_nodes=30, num_outputs=9, input_node_chance=0.4, const_min=-1, const_max=1, function_set=functions)

# Define the mutation operator with the probabilities for each type of mutation
mutationcgp_golden = Golden_mutation(CGP_config,input_node_mutation_rate=0.2, function_mutation_rate=0.4, input_mutation_rate=0.5, const_mutation_rate=0.1,output_node_mutation_rate=0.5)

# Initialize the Evolution strategy with the number of offspring with the lam parameter
ES_cgp = ES(evaluator, lam=5,parent_factory=lambda: CGPGenome.create_genome(CGP_config),mutation = mutationcgp_golden, config= CGP_config)

# Run the ES for a chosen number of generations and early stopping in case of no improvement
# Put verbose=True to see the progress of the evolution
best_genome = ES_cgp.evolve(n_generations=10000, early_stopping=10000, early_switch= 5000 ,project_name= "1027_ESL", verbose=True)

Starting fitness -10.5374


Gen 676 | Best: 0.4019:   7%|▋         | 701/10000 [01:20<17:43,  8.74gen/s]


KeyboardInterrupt: 

In [ ]:
import pandas as pd
# Load the CSV file
df = pd.read_csv('Results/cgp_results_log_1027_ESL.csv',sep=";")

# Compute the mean of a column (e.g., 'my_column')
mean_value = df['fitness'].mean()
variance_value = df['fitness'].var()

print(f"Mean of 'fitness': {mean_value}")
print(f"Variance of 'fitness': {variance_value}")

In [ ]:
## Comparison with other ML models ##
from pmlb import fetch_data
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
import pandas as pd


# --- Load the dataset ---
X, y = fetch_data('1027_ESL', return_X_y=True, local_cache_dir='./datasets')

# --- Split the dataset ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Standardize features for models that benefit from it (SVM, KNN, Ridge, etc.) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Define regression models ---
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Support Vector Regressor': SVR(),
    'KNN Regressor': KNeighborsRegressor()
}

# --- Evaluate R² for each model ---
results = {}

for name, model in models.items():
    if name in ['Support Vector Regressor', 'KNN Regressor', 'Ridge Regression']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    results[name] = r2
    print(f"{name}: R² score = {r2:.4f}")

# --- Show sorted results ---
df_results = pd.DataFrame.from_dict(results, orient='index', columns=['R2 Score'])
df_results = df_results.sort_values('R2 Score', ascending=False)
print("\n=== Sorted Results ===")
print(df_results)
print("\n=== CGP result ===")
print(f"CGP: R² score = {mean_value:.4f}")


In [ ]:
import matplotlib.pyplot as plt

# --- Add CGP result ---
df_results.loc['CGP'] = mean_value

# --- Re-sort including CGP ---
df_results = df_results.sort_values('R2 Score', ascending=False)

# --- Plot the results ---
plt.figure(figsize=(10, 6))
bars = plt.bar(df_results.index, df_results['R2 Score'], color='skyblue')
bars[df_results.index.get_loc('CGP')].set_color('orange')  # Highlight CGP

plt.title('Comparison of R² Scores Across Models')
plt.ylabel('R² Score')
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
## Initialization of the CGP algorithm for regression ##

from pmlb import fetch_data
from sklearn.preprocessing import MinMaxScaler

# Selection of dataset
# We use a PMLB dataset for regression in this example
# You can change the dataset by modifying the fetch_data function parameters and putting the Dataset ID you want to use
gametes = fetch_data('mushroom')
# We save the dataset in a local directory
X, y = fetch_data('mushroom', return_X_y=True, local_cache_dir='./datasets')

# Float conversion data
X = X.astype(float)
y = y.astype(float)

# Standardize the data between -1 and 1
# This is important for the CGP algorithm to work properly
scaler = MinMaxScaler(feature_range=(-1, 1))
X = scaler.fit_transform(X)

""" from skrebate import ReliefF
relieff = ReliefF(n_neighbors=100, n_features_to_select=10)
X = relieff.fit_transform(X, y) """

# Here we define the evaluator for the CGP algorithm
# In this case we use a regression evaluator but you can use a multi-class classifier in the case of classification problems
evaluator = Binary_Classifier(X, y, cv = True)

#Initialise config with the same number of inputs as features in the datasets, internal nodes around 20/30, outputs(number of classes)
CGP_config = CGPConfig(num_inputs=X.shape[1], num_nodes=30, num_outputs=1, input_node_chance=0.4, const_min=-1, const_max=1, function_set=functions)

# Define the mutation operator with the probabilities for each type of mutation
mutationcgp_golden = Golden_mutation(CGP_config,input_node_mutation_rate=0.2, function_mutation_rate=0.4, input_mutation_rate=0.5, const_mutation_rate=0.1,output_node_mutation_rate=0.5)

# Initialize the Evolution strategy with the number of offspring with the lam parameter
ES_cgp = ES(evaluator, lam=5,parent_factory=lambda: CGPGenome.create_genome(CGP_config),mutation = mutationcgp_golden, config= CGP_config)

# Run the ES for a chosen number of generations and early stopping in case of no improvement
# Put verbose=True to see the progress of the evolution
best_genome = ES_cgp.evolve(n_generations=1000, early_stopping=10000, early_switch= 5000 ,project_name= "mushroom", verbose=True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# --- Load the dataset ---
X, y = fetch_data('mushroom', return_X_y=True, local_cache_dir='./datasets')

# --- Split the dataset ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- Optional: Standardize features for models that benefit from it (SVM, KNN, Ridge, etc.) ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Define classification models ---
models = {
    'Logistic Regression': LogisticRegression(max_iter=100000),
    'Ridge Classifier': RidgeClassifier(),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'Support Vector Classifier': SVC(),
    'KNN Classifier': KNeighborsClassifier()
}

# --- Evaluate accuracy for each model ---
results = {}

for name, model in models.items():
    if name in ['Support Vector Classifier', 'KNN Classifier', 'Ridge Classifier']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"{name}: Accuracy = {acc:.4f}")

# --- Show sorted results ---
df_results = pd.DataFrame.from_dict(results, orient='index', columns=['Accuracy'])
df_results = df_results.sort_values('Accuracy', ascending=False)
print("\n=== Sorted Results ===")
print(df_results)


print("\n=== CGP result ===")
print(f"CGP: accuracy score = {evaluator.evaluate(best_genome)}")


In [ ]:
## Initialization of the CGP algorithm for classif ##

from pmlb import fetch_data
from sklearn.preprocessing import MinMaxScaler

# Selection of dataset
# We use a PMLB dataset for regression in this example
# You can change the dataset by modifying the fetch_data function parameters and putting the Dataset ID you want to use
gametes = fetch_data('flags')
# We save the dataset in a local directory
X, y = fetch_data('flags', return_X_y=True, local_cache_dir='./datasets')

# Float conversion data
X = X.astype(float)
y = y.astype(float)

# Standardize the data between -1 and 1
# This is important for the CGP algorithm to work properly
scaler = MinMaxScaler(feature_range=(-1, 1))
X = scaler.fit_transform(X)

""" from skrebate import ReliefF
relieff = ReliefF(n_neighbors=100, n_features_to_select=10)
X = relieff.fit_transform(X, y) """

# Here we define the evaluator for the CGP algorithm
# In this case we use a regression evaluator but you can use a multi-class classifier in the case of classification problems
evaluator = MultiClassClassifier(X, y, cv = True)

#Initialise config with the same number of inputs as features in the datasets, internal nodes around 20/30, outputs(number of classes)
CGP_config = CGPConfig(num_inputs=X.shape[1], num_nodes=30, num_outputs=5, input_node_chance=0.4, const_min=-1, const_max=1, function_set=functions)

# Define the mutation operator with the probabilities for each type of mutation
mutationcgp_golden = Golden_mutation(CGP_config,input_node_mutation_rate=0.2, function_mutation_rate=0.4, input_mutation_rate=0.5, const_mutation_rate=0.1,output_node_mutation_rate=0.5)

# Initialize the Evolution strategy with the number of offspring with the lam parameter
ES_cgp = ES(evaluator, lam=5,parent_factory=lambda: CGPGenome.create_genome(CGP_config),mutation = mutationcgp_golden, config= CGP_config)

# Run the ES for a chosen number of generations and early stopping in case of no improvement
# Put verbose=True to see the progress of the evolution
best_genome = ES_cgp.evolve(n_generations=10000, early_stopping=10000, early_switch= 5000 ,project_name= "mushroom", verbose=True)

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
# Split for evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Neural network classifier
mlp = MLPClassifier(hidden_layer_sizes=(100,), activation='relu', solver='adam',
                    max_iter=1000, random_state=42)

# Fit model
mlp.fit(X_train, y_train)

# Predict and evaluate
train_preds = mlp.predict(X_train)
test_preds = mlp.predict(X_test)

train_acc = accuracy_score(y_train, train_preds)
test_acc = accuracy_score(y_test, test_preds)

print(f"NN Train Accuracy: {train_acc:.3f}")
print(f"NN Test Accuracy: {test_acc:.3f}")
scores = cross_val_score(mlp, X, y, cv=5, scoring='accuracy')
print(f"NN CV Accuracy: {scores.mean():.3f} ± {scores.std():.3f}")


In [17]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Define the neural net regressor
mlp_reg = MLPRegressor(hidden_layer_sizes=(100,), activation='relu', solver='adam',
                       max_iter=1000, random_state=42)

# Train
mlp_reg.fit(X_train, y_train)

# Predict
train_preds = mlp_reg.predict(X_train)
test_preds = mlp_reg.predict(X_test)

# Evaluate with MSE and R²
train_mse = mean_squared_error(y_train, train_preds)
test_mse = mean_squared_error(y_test, test_preds)

train_r2 = r2_score(y_train, train_preds)
test_r2 = r2_score(y_test, test_preds)

print(f"NN Train MSE: {train_mse:.4f}, R²: {train_r2:.4f}")
print(f"NN Test MSE: {test_mse:.4f}, R²: {test_r2:.4f}")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mse_scores = []
r2_scores = []

for train_idx, val_idx in kf.split(X):
    X_fold_train, X_fold_val = X[train_idx], X[val_idx]
    y_fold_train, y_fold_val = y[train_idx], y[val_idx]

    mlp_reg.fit(X_fold_train, y_fold_train)
    val_preds = mlp_reg.predict(X_fold_val)

    mse_scores.append(mean_squared_error(y_fold_val, val_preds))
    r2_scores.append(r2_score(y_fold_val, val_preds))

print(f"NN CV Mean MSE: {np.mean(mse_scores):.4f} ± {np.std(mse_scores):.4f}")
print(f"NN CV Mean R²: {np.mean(r2_scores):.4f} ± {np.std(r2_scores):.4f}")



NN Train MSE: 0.0010, R²: 0.9961
NN Test MSE: 0.0009, R²: 0.9965
NN CV Mean MSE: 0.2919 ± 0.0530
NN CV Mean R²: 0.8537 ± 0.0247
